# 🥊 Day 02 미니 프로젝트 · RAG 기법 격돌
**여러 검색 기법을 직접 만들어 붙이고, 내 평가셋으로 숫자로 비교해 "최강 조합"을 찾습니다.**

느낌으로 "이게 더 나은 듯?" 하던 걸 이제 **Recall@k · MRR** 로 측정합니다.

### 흐름
`데이터 → 검색 기법 툴박스 → 내 평가셋 → 점수로 대결 → 최종 파이프라인 → 내 도메인 적용`

### 산출물 (제출)
1. 기법별 점수표 (기준선 vs BM25 vs 하이브리드 vs +리랭커 vs 내가 만든 기법)
2. **내가 직접 구현한 검색 기법 1개** + 왜 그렇게 만들었는지
3. 내 문서로 바꿔 재실행한 결과 + 3줄 회고
---

## 0 · 설치

In [ ]:
!pip install -q sentence-transformers qdrant-client langchain-text-splitters openai rank_bm25

## 1 · NVIDIA API 토큰 설정
생성은 NVIDIA build(Qwen)를 씁니다. 토큰이 없으면 `실습_시작하기.ipynb`에서 발급.
(검색 기법 비교는 전부 로컬이라 토큰이 필요 없고, **맨 마지막 생성 단계에서만** 씁니다.)

In [ ]:
import os, getpass
from openai import OpenAI
LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
_c = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in _c.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl","embed","rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("엔드포인트:", LLM_BASE_URL, "| 모델:", LLM_MODEL)

## 2 · 데이터 & 청킹
긴 사내 문서 하나를 통째로 넣고 청킹합니다. (`docs`를 내 문서로 바꾸면 그대로 내 프로젝트가 됩니다)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np

corpus = """[연차 및 휴가]
연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받아야 한다. 반차도 동일 절차를 따른다.
[비용 정산]
비용 정산은 영수증을 첨부해 신청하고, 경영지원팀 검토 후 재무팀이 최종 승인한다. 10만원 이상은 사전 승인이 필요하다.
[재택근무]
재택근무는 주 2회까지 가능하며 전날 18시까지 팀 채널에 사전 공유한다. 코어타임(10~16시)엔 메신저 응답을 유지한다.
[보안]
보안상 외부 USB 사용은 금지이며 파일 공유는 사내 드라이브를 사용한다. 외부 공유는 보안팀 사전 승인이 필요하다.
[오류 코드]
에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다. ERR-5010은 네트워크 문제로 VPN을 확인한다.
[제품 X-200]
제품 X-200의 펌웨어 최신 버전은 2.3이며 설정 메뉴에서 업데이트한다. 배터리는 파워셀이 공급한다.
[온보딩]
신입은 입사 첫날 계정과 장비를 받고 보안 서약서를 작성한다. 1주일 내 정보보안 교육을 이수해야 한다."""

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_text(corpus)
print(len(chunks), "청크")

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
def embed(xs): return embedder.encode(xs, normalize_embeddings=True)
cvecs = embed(chunks)

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
qc = QdrantClient(":memory:")
qc.recreate_collection("proj", vectors_config=VectorParams(size=cvecs.shape[1], distance=Distance.COSINE))
qc.upsert("proj", points=[PointStruct(id=i, vector=cvecs[i].tolist(), payload={"text": chunks[i]}) for i in range(len(chunks))])
print("임베딩·색인 완료:", cvecs.shape)

## 3 · 검색 기법 툴박스
각 기법은 **질문 → 청크 인덱스 순위 리스트**를 반환하는 함수입니다. 나중에 이걸 갈아끼우며 비교합니다.

In [ ]:
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# (1) 벡터 검색
def retrieve_vector(query, k=3):
    qv = embed([query])[0]
    return [p.id for p in qc.query_points("proj", query=qv.tolist(), limit=k).points]

# (2) BM25 (한글 조사 분리 토크나이저)
def tok(t): return re.findall(r"[A-Za-z0-9\-]+|[가-힣]+", t)
bm25 = BM25Okapi([tok(c) for c in chunks])
def retrieve_bm25(query, k=3):
    s = bm25.get_scores(tok(query))
    return sorted(range(len(s)), key=lambda i: -s[i])[:k]

# (3) 하이브리드 (RRF 융합)
def rrf(rank_lists, k=60):
    score = {}
    for ranks in rank_lists:
        for r, idx in enumerate(ranks):
            score[idx] = score.get(idx, 0) + 1/(k + r + 1)
    return sorted(score, key=lambda i: -score[i])
def retrieve_hybrid(query, k=3):
    return rrf([retrieve_vector(query, 10), retrieve_bm25(query, 10)])[:k]

# (4) 하이브리드 + 리랭커 (다국어 cross-encoder)
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
def retrieve_rerank(query, k=3):
    cand = retrieve_hybrid(query, 8)
    scores = reranker.predict([(query, chunks[i]) for i in cand])
    return [cand[j] for j in sorted(range(len(cand)), key=lambda j: -scores[j])[:k]]

print("기법 4개 준비: vector / bm25 / hybrid / rerank")

## 4 · [내가 만들기] 평가셋
검색이 "잘 됐다"를 측정하려면 정답이 필요합니다. **질문 + 정답이 들어있어야 할 키워드** 쌍을 만드세요.
(청크에 그 키워드가 있으면 '정답 청크'로 칩니다.) 아래 5개는 예시 — **내 문서에 맞게 8~10개로 늘리세요.**

In [ ]:
EVALSET = [
    ("연차는 며칠 전에 신청하나요?",        "3영업일"),
    ("비용은 누가 최종 승인해요?",          "재무팀"),
    ("인증 토큰이 만료되면 뜨는 코드는?",   "ERR-4021"),
    ("X-200 펌웨어 최신 버전은?",           "2.3"),
    ("재택 언제까지 공유해야 하나요?",       "전날 18시"),
    # TODO: 내 문서에 맞는 질문-정답 키워드 쌍을 더 추가하세요
]
print(len(EVALSET), "개 평가 항목")

## 5 · 평가 하네스 (Recall@k · MRR)
- **Recall@k**: top-k 안에 정답 청크가 하나라도 있으면 성공 → 성공한 질문 비율
- **MRR**: 정답 청크가 몇 등에 처음 나왔는지(1등=1.0, 2등=0.5…)의 평균

In [ ]:
def evaluate(retriever_fn, evalset=EVALSET, k=3):
    recall, mrr = 0, 0.0
    for q, ans_kw in evalset:
        idxs = retriever_fn(q, k)
        hit_ranks = [r for r, i in enumerate(idxs) if ans_kw in chunks[i]]
        if hit_ranks:
            recall += 1
            mrr += 1 / (hit_ranks[0] + 1)
    n = len(evalset)
    return {"Recall@%d" % k: round(recall/n, 3), "MRR": round(mrr/n, 3)}

print("예시:", evaluate(retrieve_vector))

## 6 · 🥊 기법 대결 [실행 + 메모]
같은 평가셋으로 4개 기법을 돌려 점수표를 만듭니다. **어느 기법이 왜 이겼는지 옆에 메모**하세요.

In [ ]:
methods = {
    "① 벡터(기준선)": retrieve_vector,
    "② BM25":        retrieve_bm25,
    "③ 하이브리드":   retrieve_hybrid,
    "④ +리랭커":      retrieve_rerank,
}
print(f"{'기법':<16}{'Recall@3':>10}{'MRR':>8}")
print("-" * 34)
for name, fn in methods.items():
    r = evaluate(fn)
    print(f"{name:<16}{r['Recall@3']:>10}{r['MRR']:>8}")

## 7 · [도전] 새 검색 기법을 직접 만들어 대결에 추가
아래는 **예시 기법**(3개 검색 결과를 RRF로 앙상블)입니다. 이걸 참고해 **여러분만의 기법**을 만들어 위 표에 넣어보세요.

아이디어:
- **쿼리 확장**: 질문에 동의어/키워드를 덧붙여 검색
- **길이 가중치**: 너무 짧은 청크에 페널티
- **메타데이터 필터**: 제목(`[오류 코드]` 등)으로 먼저 거르기
- **2단계**: BM25로 넓게 → 벡터로 재정렬

In [ ]:
# 예시: 벡터+BM25+리랭커 3개를 RRF로 앙상블
def retrieve_ensemble(query, k=3):
    cand = rrf([retrieve_vector(query, 8), retrieve_bm25(query, 8), retrieve_rerank(query, 8)])[:k]
    return cand

# TODO: 아래에 내 기법을 구현하고 methods 표에 추가해 점수를 비교하세요
def retrieve_myown(query, k=3):
    # 예) 쿼리 확장 후 하이브리드
    expanded = query + " " + " ".join(tok(query))   # 아주 단순한 확장(자유롭게 개선)
    return retrieve_hybrid(expanded, k)

for name, fn in {"⑤ 앙상블": retrieve_ensemble, "⑥ 내 기법": retrieve_myown}.items():
    print(f"{name:<16}", evaluate(fn))

## 8 · 최종 RAG 파이프라인 (최고 기법 → 생성)
점수가 가장 좋았던 검색기를 골라 **검색→근거→생성**까지 이어붙입니다. (여기서만 LLM 호출)

In [ ]:
BEST = retrieve_rerank   # ← 위 표에서 가장 좋았던 기법으로 바꾸세요

SYSTEM = ("아래 [자료]의 내용만 근거로 답하라. 없으면 '문서에 없음'이라고만 답하라. 답 끝에 (근거)를 표기하라.")
def final_rag(query, k=3):
    idxs = BEST(query, k)
    ctx = "\n".join(f"[{i+1}] {chunks[d]}" for i, d in enumerate(idxs))
    out = _c.chat.completions.create(model=LLM_MODEL, temperature=0.2, max_tokens=200,
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":f"[자료]\n{ctx}\n\n[질문] {query}"}])
    return out.choices[0].message.content.split("</think>")[-1].strip()

for q in ["연차 며칠 전에 신청해요?", "X-200 펌웨어 어떻게 올려요?", "사내 헬스장 운영시간은?"]:
    print("Q:", q, "\nA:", final_rag(q), "\n" + "-"*40)

## 9 · [내 도메인 적용 + 회고]
- 위 `corpus`를 **내 실제 문서**로 바꾸고, `EVALSET`도 내 질문-정답으로 바꿔 6·7·8번을 다시 실행하세요.
- 회고 3줄: ① 어떤 기법이 왜 최고였나 ② 내가 만든 기법은 효과가 있었나 ③ 어떤 질문에서 실패했나

## ✅ 제출물 체크리스트
- [ ] 기법별 점수표 (①~⑥) + 승자 메모
- [ ] 내가 직접 구현한 검색 기법(`retrieve_myown`) + 설계 이유
- [ ] 내 문서로 바꿔 재실행한 결과 + 3줄 회고
- [ ] 최종 파이프라인이 '문서에 없는 질문'에 '문서에 없음'으로 답하는지 확인

> 💡 이 프로젝트의 `BEST` 검색기가 Day03~05에서 **에이전트/MCP의 검색 도구**로 연결됩니다.